# Atelier Prompt Engineering

## Partie 1 – Anatomie d'un prompt 

Construire un prompt, à partir de ses différentes composantes, pour résoudre un problème comme « Je souhaite analyser les retours de clients d'une entreprise. » 

![prompt Partie 1](images/partie1_prompt_1.png)
![prompt Partie 1](images/partie1_prompt_2.png)
![Réponse Partie 1](images/partie1_reponse.png)

## Partie 2 - Comparer les techniques de prompting

Tester puis évaluer les résultats obtenus avec les techniques zero-shot, one-shot, few-shot et un prompt structuré sur une demande du genre : «  Classer le commentaire suivant :   "Le service est rapide mais l'application plante régulièrement."   Classes possibles :    positif, négatif, neutre. » 

![Prompt Partie 2 avec zero-shot](images/partie2_zero-shot.png)

Réponse zero-shot : "Négatif", avec une justification spontanée (le LLM explique pourquoi malgré l'aspect positif "rapide"). Intéressant même sans qu'on le demande, il a choisi de justifier, ce sera à comparer avec les autres techniques.

![prompt Partie 2 avec one-shot](images/partie2_one-shot.png)

Le format a changé comme prévu : le LLM a imité le style de l'exemple donné ("Classe : X", court, sans explication) ça montre bien l'effet du one-shot sur la forme de la réponse.

![Prompt Partie 2 avec few-shot](images/partie2_few-shot.png)

Avec plusieurs exemples bien répartis entre les 3 classes (positif, négatif, neutre), le LLM revient sur "négatif", cohérent avec le zero-shot, mais cette fois sans justification (il a gardé le format court de l'exemple). Ça confirme l'hypothèse : le one-shot avait biaisé la réponse vers "neutre" à cause d'un exemple non représentatif, alors que le few-shot, avec des exemples plus variés, permet un jugement plus fiable* proche de l'intuition naturelle du zero-shot, mais avec un format plus contrôlé.

![Prompt Partie 2 avec structured prompting](images/partie2_structured-prompting.png)

| Technique | Classe | Justification | Format |
|---|---|---|---|
| Zero-shot | Négatif | Oui (spontanée) | Libre, développé |
| One-shot | Neutre | Non | Court, imité de l'exemple |
| Few-shot | Négatif | Non | Court, imité de l'exemple |
| Structuré | Négatif | Oui (demandée) | Précis, imposé |

## Évaluation
### Stabilité de la classification :
3 techniques sur 4 (zero-shot, few-shot, structuré) convergent vers "négatif". Seul le one-shot a dévié vers "neutre" probablement biaisé par l'unique exemple fourni ("cher mais bien fait" → neutre), qui n'était pas assez représentatif du cas à traiter. Leçon : un seul exemple mal choisi peut fausser le jugement du LLM davantage qu'aucun exemple du tout.
### Contrôle du format :
le zero-shot laisse le LLM libre (il choisit de justifier). Le one-shot et le few-shot imposent implicitement un format via l'exemple (réponse courte). Le prompt structuré est le seul à garantir explicitement le format ET la présence d'une justification, sans dépendre d'un exemple.
### Fiabilité générale :
le prompt structuré est la technique la plus fiable ici — il combine la justesse du zero-shot/few-shot (bonne classe) avec un format prévisible et une justification garantie, contrairement au one-shot qui s'est révélé le moins fiable des quatre sur ce cas précis.
### Cas ambigu révélateur :
ce commentaire mixte (aspect positif + négatif) était un bon choix de test il a permis de révéler que le nombre et la qualité des exemples influencent directement le jugement du LLM, pas seulement la forme de sa réponse.

## Partie 3 – Prompt Engineering et raisonnement

### 1) Décomposer le prompt suivant : 
« Analyse ces avis clients et donne-moi les problèmes les plus importants ainsi que les recommandations. » 

### Prompt

Analyse les avis clients suivants en suivant ces étapes :

1. Lis attentivement chaque avis client fourni.
2. Identifie les problèmes mentionnés dans chaque avis.
3. Regroupe les problèmes similaires et classe-les par ordre d'importance (fréquence ou gravité).
4. Pour chaque problème important identifié, propose une recommandation concrète.

Présente ta réponse en suivant ces 4 étapes, dans l'ordre, en indiquant clairement le résultat de chaque étape.

Avis clients à analyser :
{"input": "Livraison très en retard, aucune information pendant 5 jours, très déçu.", "output": 2}
{"input": "L'application plante souvent quand je veux payer, c'est frustrant.", "output": 3}
{"input": "Service client réactif et sympathique, ils ont résolu mon problème en 10 minutes.", "output": 9}
{"input": "Produit conforme à la description, rien à redire.", "output": 8}
{"input": "J'ai attendu 4 jours une réponse du support pour ma réclamation.", "output": 3}
{"input": "Très satisfait de la qualité, je recommande sans hésiter.", "output": 10}
{"input": "Le site web est lent et l'interface pas très claire.", "output": 4}
{"input": "Bon rapport qualité-prix, livraison rapide cette fois-ci.", "output": 8}
{"input": "Encore un bug sur l'application, ça devient récurrent.", "output": 2}
{"input": "Commande arrivée à temps, emballage soigné, rien à signaler.", "output": 9}

### Reponse

### Étape 1 — Lecture des avis et problèmes mentionnés

| Avis | Problème identifié |
|---|---|
| Livraison très en retard, aucune information pendant 5 jours | Retard de livraison + manque d’information |
| L’application plante lors du paiement | Bug de l’application lors du paiement |
| Service client réactif | Aucun problème signalé |
| Produit conforme | Aucun problème signalé |
| Réponse du support après 4 jours | Délai de réponse du support |
| Client très satisfait | Aucun problème signalé |
| Site lent et interface peu claire | Lenteur du site + interface peu claire |
| Livraison rapide | Aucun problème signalé |
| Bug récurrent de l’application | Bugs récurrents de l’application |
| Commande à temps et emballage soigné | Aucun problème signalé |

---

### Étape 2 — Regroupement des problèmes similaires

1. **Problèmes liés à l’application** — 2 avis
   - Plantage lors du paiement.
   - Bugs récurrents.

2. **Problèmes liés à la livraison et à la communication** — 1 avis
   - Livraison très en retard.
   - Absence d’information pendant 5 jours.

3. **Problèmes liés au support client** — 1 avis
   - Réponse après 4 jours.

4. **Problèmes liés au site web** — 1 avis
   - Site lent.
   - Interface peu claire.

---

### Étape 3 — Classement par importance

1. **Instabilité de l’application** — 2 avis, dont un problème affectant directement le paiement.
2. **Retard et manque de suivi des livraisons** — problème important en termes d’expérience client.
3. **Délais de réponse du support** — 4 jours pour une réclamation.
4. **Lenteur et manque de clarté du site** — problème d’utilisation mentionné dans un avis.

---

### Étape 4 — Recommandations concrètes

1. **Application**
   - Prioriser l’analyse et la correction des bugs, notamment ceux qui provoquent des plantages pendant le paiement.
   - Mettre en place des tests réguliers pour éviter la réapparition des bugs.

2. **Livraison**
   - Améliorer le suivi des commandes et informer automatiquement le client en cas de retard.
   - Éviter les périodes prolongées sans information sur l’état de la livraison.

3. **Support client**
   - Réduire le délai de traitement des réclamations et assurer un suivi plus rapide des demandes.

4. **Site web**
   - Optimiser les performances du site et clarifier l’interface afin de faciliter son utilisation.

### 2) Proposer au modèle un texte à analyser avec des contraintes puis utiliser un second prompt lui demandant de vérifier sa réponse précédente et de contrôler les informations non justifiées, les contradictions, les informations absentes, les éventuelles hallucinations et le respect des contraintes. 

## Prompt

Vérifie la réponse précédente à l'analyse des avis clients en répondant aux points suivants :

1. Y a-t-il des informations non justifiées par les avis clients fournis ?
2. Y a-t-il des contradictions dans l'analyse ?
3. Y a-t-il des informations importantes absentes ou oubliées ?
4. Y a-t-il des hallucinations (informations inventées, absentes des avis d'origine) ?
5. Les contraintes initiales ont-elles été respectées (rester factuel, ne pas inventer, structurer par étapes) ?

Réponds point par point pour chacun de ces 5 axes.

## Reponse

## 1. Informations non justifiées par les avis clients

**Oui, légèrement.**

- La phrase **« problème important en termes d’expérience client »** est une interprétation générale. Elle est plausible, mais elle n'est pas directement exprimée dans les avis.
- La recommandation de **« prioriser »** certains problèmes constitue une proposition d'analyse et non un fait directement fourni par les clients.
- Le reste des informations est globalement directement justifié par les commentaires.

## 2. Contradictions dans l’analyse

**Non, il n’y a pas de contradiction majeure.**

- Les deux avis concernant l’application signalent bien des problèmes : plantage au paiement et bugs récurrents.
- Les avis positifs sur la livraison (« livraison rapide », « commande arrivée à temps ») ne contredisent pas le retard signalé dans un autre avis. Ils montrent simplement des expériences différentes.
- Le support client présente toutefois des expériences contrastées : un client indique une résolution en 10 minutes, tandis qu’un autre a attendu 4 jours pour une réponse.

## 3. Informations importantes absentes ou oubliées

**Oui, quelques éléments peuvent être améliorés.**

- Il aurait été pertinent de signaler explicitement que **les expériences concernant le support client sont contrastées**.
- Les avis positifs concernant la **qualité et la conformité des produits** sont peu exploités dans l'analyse.
- Il aurait également été utile de préciser que **les problèmes liés à l’application sont les seuls problèmes mentionnés dans plusieurs avis**, avec 2 avis concernés.

## 4. Hallucinations

**Il n’y a pas d’hallucination importante.**

Les principaux éléments correspondent bien aux avis fournis :

- **5 jours sans information** → présent dans l’avis sur la livraison.
- **4 jours d’attente du support** → présent dans l’avis concernant la réclamation.
- **2 problèmes liés à l’application** → présents dans deux avis différents.
- **Site lent et interface peu claire** → présents dans l’avis correspondant.
- **Livraison rapide ou à temps** → présentes dans les avis positifs.

Les recommandations comme **« mettre en place des tests réguliers »** ou **« informer automatiquement le client »** sont des propositions et non des informations présentées comme des faits. Elles ne constituent donc pas des hallucinations.

## 5. Respect des contraintes initiales

**Oui, globalement les contraintes ont été respectées.**

- **Rester factuel :** globalement respecté.
- **Ne pas inventer :** respecté ; aucune donnée externe ou statistique inventée n’a été ajoutée.
- **Identifier les problèmes :** respecté.
- **Regrouper les problèmes similaires :** respecté.
- **Classer les problèmes par importance :** respecté, même si le classement pourrait être davantage justifié par la fréquence et la gravité.
- **Proposer des recommandations concrètes :** respecté.
- **Structurer la réponse en 4 étapes :** respecté.

## Conclusion

La réponse est **globalement correcte et fidèle aux données fournies**.

Les principales améliorations seraient de :

1. mieux faire ressortir les **avis positifs** ;
2. signaler explicitement la **divergence concernant le support client** ;
3. distinguer plus clairement les **faits observés** des **interprétations et recommandations**.

## Partie 4 – Sorties structurées

### Le LLM retourne : « Le commentaire semble plutôt négatif. Le client est mécontent du délai de livraison... » 
#### 1) Pour rendre cette réponse plus facile à exploiter par une application, créer un prompt demandant du JSON avec les champs sentiment, categorie, urgence, probleme, confiance. 
Préciser les types des valeurs et les valeurs autorisées
Exemple de résultat 
```json
{
    "sentiment": "negatif",
    "categorie": "livraison",
    "urgence": "moyenne", 
    "probleme": "Retard de livraison", 
    "confiance": 0.91 
}
```

### Prompt

Analyse le commentaire client suivant et retourne le résultat **uniquement** au format JSON, avec les champs suivants :

- `sentiment` (string) : "positif", "negatif" ou "neutre"
- `categorie` (string) : le thème principal du commentaire (ex. "livraison", "application", "support", "produit")
- `urgence` (string) : "faible", "moyenne" ou "élevée"
- `probleme` (string) : description courte du problème principal identifié
- `confiance` (number) : score entre 0 et 1 représentant ton degré de certitude dans cette analyse

Ne retourne rien d'autre que le JSON.

Commentaire à analyser : "Le commentaire semble plutôt négatif. Le client est mécontent du délai de livraison..."

### Reponse

```json
{
  "sentiment": "negatif",
  "categorie": "livraison",
  "urgence": "moyenne",
  "probleme": "Délai de livraison jugé insatisfaisant.",
  "confiance": 0.98
}
```

#### 2) Ajouter une validation en définissant les règles de sortie suivantes : format JSON valide ; aucune propriété supplémentaire ; sentiment vaut positif, negatif ou neutre ; confiance est compris entre 0 et 1 ; urgence vaut faible, moyenne ou élevée. 

### Prompt

Analyse le commentaire client suivant et retourne le résultat *uniquement* au format JSON, en respectant strictement les règles suivantes :

Le JSON doit être valide (syntaxe correcte, pas de texte autour).
Aucune propriété supplémentaire ne doit être ajoutée en dehors des 5 champs suivants.
- `sentiment` (string) : doit valoir exactement "positif", "negatif" ou "neutre".
- `categorie` (string) : thème principal du commentaire.
- `urgence` (string) : doit valoir exactement "faible", "moyenne" ou "élevée".
- `probleme` (string) : description courte du problème principal.
- `confiance` (number) : doit être compris entre 0 et 1.

Commentaire à analyser : "Le commentaire semble plutôt négatif. Le client est mécontent du délai de livraison..."

### Reponse

{"sentiment":"negatif","categorie":"livraison","urgence":"moyenne","probleme":"Délai de livraison insatisfaisant.","confiance":0.99}

#### Comparaison des deux versions

| Aspect | Version 1 (sans règles) | Version 2 (avec règles) |
|---|---|---|
| Format | JSON indenté (lisible) | JSON sur une ligne (compact) |
| Champs | 5, corrects | 5, corrects |
| Confiance	| 0.98	| 0.99 |
| Propriétés en trop | Aucune | Aucune |

Résultat : JSON valide, format compact, tous les champs corrects, confiance 0.99.

À retenir :

Le format a changé (indenté → compact) sans qu'on l'ait demandé, si on veut un format visuel stable, il faut le préciser explicitement.
Le score de confiance reste très élevé (0.98→0.99) sans méthode de calcul claire, à noter comme limite (indicatif, pas vraiment fiable).
Les règles de validation (pas de champ en trop, valeurs autorisées) sont respectées dans les deux versions.